---
#Colab solo para pruebas
---

In [ ]:
print("Hola Mundo!!")

Hola Mundo!!


In [ ]:
"""
Datos diarios en las 16 estaciones del Litoral desde ERA5 (reanálisis del
ECMWF) vía la API gratuita de Open-Meteo. Se actualiza con ~5 días de demora.

NOAA GSOD y Meteostat todavía no tienen 2026 para estas estaciones; ERA5 sí.
OJO: ERA5 no es la medición del pluviómetro sino una grilla de ~25 km. Por eso
el script baja TAMBIÉN ago-oct de 1991-2020 de ERA5 en los mismos puntos: las
anomalías de 2026 se calculan contra la climatología de ERA5, no contra la del SMN.

Uso en Colab:   !python descargar_litoral_era5.py
Salida (mismo formato que smn_datos_meteorologicos_desde_2021.lst), ago-oct de
1991-2020 y de 2026:
  era5_litoral_ago_oct.lst  (+ .csv)
"""
import os, time, datetime as dt
import numpy as np, pandas as pd, requests

HASTA = (dt.date.today() - dt.timedelta(days=1)).isoformat()
# Solo ago-oct: de 2026 (lo que se analiza) y de 1991-2020 (para la climatología
# ERA5 de esos mismos meses, contra la que se calculan las anomalías de 2026)
MESES = ("08-01", "10-31")
TRAMOS = [(f"{a}-{MESES[0]}", f"{a}-{MESES[1]}") for a in range(1991, 2021)] + \
         [(f"2026-{MESES[0]}", HASTA)]
ESTACIONES = {  # NRO_OMM: (NOMBRE, lat, lon) de smn_estaciones_meteorologicas21-26.xlsx
    87257: ("CERES AERO", -29.87556, -61.93604),
    87270: ("RECONQUISTA AERO", -29.20508, -59.69338),
    87371: ("SAUCE VIEJO AERO", -31.70780, -60.80532),
    87468: ("VENADO TUERTO", -33.68226, -61.96136),
    87480: ("ROSARIO AERO", -32.90816, -60.78216),
    87470: ("EL TRÉBOL", -32.19216, -61.71198),
    87374: ("PARANÁ AERO", -31.79033, -60.48411),
    87395: ("CONCORDIA AERO", -31.30223, -58.00147),
    87497: ("GUALEGUAYCHÚ AERO", -33.01245, -58.61076),
    87166: ("CORRIENTES AERO", -27.44979, -58.75974),
    87289: ("PASO DE LOS LIBRES AERO", -29.68931, -57.14849),
    87393: ("MONTE CASEROS AERO", -30.26172, -57.63900),
    87097: ("IGUAZÚ AERO", -25.73108, -54.47897),
    87163: ("BERNARDO DE IRIGOYEN AERO", -26.27793, -53.67121),
    87178: ("POSADAS AERO", -27.39100, -55.96705),
    87187: ("OBERÁ", -27.48372, -55.13158),
}
URL = "https://archive-api.open-meteo.com/v1/archive"
# Pocas variables: Open-Meteo cobra "peso" por años x variables y limita por hora
VARS = ["temperature_2m_max", "temperature_2m_min", "temperature_2m_mean", "precipitation_sum"]
CACHE = "cache_era5"  # cada estación bajada se guarda acá y no se vuelve a pedir
os.makedirs(CACHE, exist_ok=True)

def bajar(omm, nombre, lat, lon):
    """Devuelve (DataFrame, None) o (None, motivo). Guarda en CACHE lo bajado."""
    arch = f"{CACHE}/{omm}_{HASTA}.csv"
    if os.path.exists(arch):
        return pd.read_csv(arch, parse_dates=["time"]), None
    partes = []
    for desde, hasta in TRAMOS:  # 31 pedidos chicos (~92 días cada uno)
        params = dict(latitude=lat, longitude=lon, start_date=desde, end_date=hasta,
                      daily=",".join(VARS), models="era5",
                      timezone="America/Argentina/Buenos_Aires")
        for intento in range(3):
            r = requests.get(URL, params=params, timeout=120)
            if r.status_code != 429 or "Hourly" in r.text or "Daily" in r.text:
                break
            time.sleep(60)  # límite por minuto: esperar y reintentar
        if r.status_code == 429:
            return None, "limite"
        if r.status_code != 200:
            return None, f"HTTP {r.status_code} {r.text[:200]}"
        partes.append(pd.DataFrame(r.json()["daily"]))
        time.sleep(0.5)
    g = pd.concat(partes, ignore_index=True)
    g["time"] = pd.to_datetime(g["time"])
    g.to_csv(arch, index=False)
    return g, None

filas, pendientes = [], []
for omm, (nombre, lat, lon) in ESTACIONES.items():
    if pendientes and not os.path.exists(f"{CACHE}/{omm}_{HASTA}.csv"):
        pendientes.append(nombre); continue  # límite agotado: no insistir
    g, error = bajar(omm, nombre, lat, lon)
    if error == "limite":
        print(f"{omm} {nombre}: límite por hora de Open-Meteo alcanzado")
        pendientes.append(nombre); continue
    if error:
        print(f"{omm} {nombre}: {error}"); continue
    print(f"{omm} {nombre}: ok")
    filas.append(pd.DataFrame({
        "NRO_OMM": omm, "NOMBRE": nombre, "FECHA": g["time"].dt.strftime("%d/%m/%Y"),
        "TMAX": g["temperature_2m_max"], "TMIN": g["temperature_2m_min"],
        "TMEDIA": g["temperature_2m_mean"], "PRECIP": g["precipitation_sum"],
        "HELIOF": np.nan, "VTO_MDIR": np.nan, "VTO_MVEL": np.nan, "VEL_MED": np.nan,
        "PRE_EST": np.nan, "HUM_REL": np.nan, "NUB_TOT": np.nan,
        "FUENTE": "ERA5",
    }))

if pendientes:
    raise SystemExit(f"\nFaltan {len(pendientes)} estaciones: {', '.join(pendientes)}.\n"
                     "Volvé a correr el script en una hora: las ya bajadas quedan guardadas en "
                     f"'{CACHE}/' y solo pide las que faltan.")
if not filas:
    raise SystemExit("No se bajó ninguna estación. Pasame los mensajes de arriba.")
ult = pd.to_datetime(pd.concat(filas).dropna(subset=["PRECIP"]).FECHA, format="%d/%m/%Y").max()
print(f"Último día con lluvia ERA5: {ult:%d/%m/%Y}")

d = pd.concat(filas, ignore_index=True)
d = d.dropna(subset=["TMAX", "TMIN", "PRECIP"], how="all")  # días que ERA5 aún no publicó
num = [c for c in d.columns if c not in ("NRO_OMM", "NOMBRE", "FECHA", "FUENTE")]
d[num] = d[num].round(1).astype(object).where(d[num].notna(), "S/D")
d.to_csv("era5_litoral_ago_oct.lst", sep="\t", index=False, encoding="latin-1")
d.to_csv("era5_litoral_ago_oct.csv", index=False, encoding="utf-8")

# Resumen: lluvia regional de 2026 contra la climatología ERA5 1991-2020
m = d.assign(F=pd.to_datetime(d.FECHA, format="%d/%m/%Y"),
             PP=pd.to_numeric(d.PRECIP, errors="coerce"))
mens = m.groupby(["NRO_OMM", m.F.dt.year.rename("anio"), m.F.dt.month.rename("mes")]).PP.sum(min_count=25)
clim = mens.loc[(slice(None), slice(1991, 2020))].groupby(["NRO_OMM", "mes"]).agg(["mean", "std"])
z = ((mens - clim["mean"]) / clim["std"]).dropna()
print("\nAnomalía estandarizada de lluvia (z) promedio de las estaciones, 2026 (ERA5 vs ERA5 1991-2020):")
print(z.loc[(slice(None), 2026)].groupby("mes").mean().round(2).to_string())
print(f"\n{len(d)} filas guardadas en era5_litoral_ago_oct.lst")


87257 CERES AERO: límite por hora de Open-Meteo alcanzado


SystemExit: 
Faltan 16 estaciones: CERES AERO, RECONQUISTA AERO, SAUCE VIEJO AERO, VENADO TUERTO, ROSARIO AERO, EL TRÉBOL, PARANÁ AERO, CONCORDIA AERO, GUALEGUAYCHÚ AERO, CORRIENTES AERO, PASO DE LOS LIBRES AERO, MONTE CASEROS AERO, IGUAZÚ AERO, BERNARDO DE IRIGOYEN AERO, POSADAS AERO, OBERÁ.
Volvé a correr el script en una hora: las ya bajadas quedan guardadas en 'cache_era5/' y solo pide las que faltan.

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Pasos del análisis: de los datos al informe final

**Pregunta del TP:** A octubre de 2026, ¿podemos afirmar que el Litoral argentino ya está bajo los efectos de El Niño?

**Respuesta final:** sí, pero con matices. En el océano, El Niño está declarado y es récord. La atmósfera ya acopla (el SOI es negativo). En el Litoral, las lluvias de julio a septiembre fueron altas, pero el efecto fuerte de El Niño sobre la región se da en octubre, noviembre y diciembre, que recién empieza. Las crecidas del Iguazú y del Uruguay vienen de las lluvias en las cuencas altas de Brasil.

---

## 1. Juntar los datos

| Fuente | Qué aporta | Archivo |
|---|---|---|
| Dataset del Pacífico (1950–2026) | ONI, Niño 3.4, RONI, SOI y otros índices ENSO | `TP-Labo/DF/dataset_pacifico_final_limpio.csv` |
| SMN 1991–2020 | El período de referencia climatológica ("lo normal") | `TP-Labo/DF/smn_datos_meteorologicos_1991_2020.xlsx` (también en `.csv`) |
| SMN 2021 a julio de 2026 | Datos observados recientes | `smn_datos_meteorologicos_desde_2021.lst` |
| NASA POWER | Datos de agosto a octubre de 2026, que el SMN todavía no publicó | `datos_sep_oct_2026/nasapower_litoral_ago_oct.csv` |
| Tesis de Robledo | Método de los índices P75 y PE75 (lluvias intensas) | — |
| NOAA CPC | Valores oficiales de 2026 y aviso de El Niño | Páginas web del CPC |

Se probaron otras fuentes que no sirvieron:
- **NOAA GSOD y Meteostat:** no tenían datos de 2026.
- **Open-Meteo/ERA5:** devolvía errores por exceso de pedidos (límite por hora).

Por eso se usó NASA POWER.

## 2. Validar NASA contra el SMN

Se comparó NASA con el SMN en el período 1991–2020, para las 16 estaciones del Litoral. La concordancia es alta (r ≈ 0,95–0,98 en los totales de lluvia). Conclusión: NASA sirve para completar los meses faltantes.

## 3. Combinar SMN y NASA

- **Para el análisis:** se usó el SMN hasta julio de 2026 y NASA solo para los días que el SMN no tiene. La unión se hizo con **anomalías estandarizadas (z)** y no con milímetros, para no mezclar escalas.
- **Para el archivo combinado que pediste:** se convirtieron los datos de NASA a la escala del SMN.
  - Lluvia: *quantile mapping*, calibrado con agosto a octubre de 1991–2020.
  - Temperaturas, humedad y presión: corrección del sesgo medio.
  - Viento: corrección por cociente.
  - Una columna `FUENTE` indica si cada fila viene del SMN o de NASA ajustado.
  - Archivo: `datos_sep_oct_2026/smn_datos_meteorologicos_desde_2021_con_nasa.csv`.

## 4. Calcular los indicadores regionales

- **Climatología:** media y desvío mensual de cada estación en 1991–2020.
- **Anomalía z:** (valor − media) / desvío, por estación. Después se promedió entre las estaciones del Litoral.
- **P75 / PE75** (tesis de Robledo):
  - P75: percentil 75 de la lluvia diaria de cada estación (entre 14 y 26 mm).
  - PE75: cuántos días superan ese umbral.
- **Relación histórica ENSO–lluvia:**
  - Correlación entre el ONI y la lluvia del Litoral por trimestre. En octubre a diciembre es r = 0,76.
  - PE75 según la fase del ENSO, con test de Mann-Whitney (p = 0,02).

## 5. Corregir el dataset del Pacífico ⚠️

Los valores de 2026 del dataset original estaban mal: no mostraban El Niño. Con ellos, una primera versión del análisis concluyó erróneamente que no había El Niño. Se reemplazaron por los valores oficiales de NOAA:
- ONI de julio a septiembre (JAS): +2,16.
- Niño 3.4 de septiembre: +2,56.
- RONI.

El dataset original se guardó como `dataset_pacifico_final_limpio_ORIGINAL.csv`, y los cambios están en `CAMBIOS_dataset_pacifico_2026.md`.

## 6. Entender por qué los medios ya hablaban de El Niño declarado

NOAA usa dos definiciones:
- **Operativa:** el aviso de El Niño (*El Niño Advisory*), vigente desde el 11/06/2026.
- **Histórica:** requiere 5 trimestres móviles seguidos con ONI ≥ +0,5.

Los medios siguen la definición operativa, que sale antes.

## 7. Analizar las crecidas (Iguazú y Litoral)

- Se armó un dataset de 14 puntos en las cuencas altas del sur de Brasil (`datos_sep_oct_2026/cuencas_altas_*.csv`).
- Iguazú alto: la lluvia de julio a septiembre fue un 64 % superior a lo normal (la 3.ª más alta desde 1991).
- Uruguay alto: del 1 al 12 de octubre llovió un 240 % de lo normal (3.º lugar, detrás de 1997 y 2023).
- Las crecidas vienen sobre todo de lluvias aguas arriba, no de lo que llovió en el Litoral mismo.

## 8. Hacer los gráficos

Los scripts están en `analisis_el_nino/scripts/`:
1. Línea de tiempo del ONI (1950–2026).
2. Panel 2023–2026: ONI comparado con la lluvia del Litoral. Las barras rayadas son datos de NASA.
3. Trayectoria del ONI en 2026 comparada con los El Niño fuertes de 1997, 2009, 2015 y 2023.
4. Lluvia en las cuencas altas en 2026.

## 9. Redactar los informes

- `informe_el_nino/informe_respuesta_TP.md`: el informe final, versión 2. Incluye:
  - el estado del océano;
  - el SOI;
  - la respuesta regional;
  - las crecidas;
  - una tabla de síntesis;
  - las conclusiones y limitaciones;
  - una reflexión metodológica sobre el error del dataset.
- `analisis_el_nino/analisis_variables_graficos_limitaciones.md`: el análisis de variables, las hipótesis H1 a H6 y las limitaciones.
- Las versiones anteriores, hechas con los datos erróneos, quedaron en las carpetas `version_anterior_datos_erroneos/`.

## Limitaciones principales

- Los datos de agosto a octubre de 2026 son de NASA ajustado, no observaciones del SMN.
- El efecto más fuerte de El Niño en el Litoral (octubre a diciembre) recién empieza.
- Los valores de 2026 del dataset del Pacífico se cargaron a mano desde NOAA.
- Algunos índices de 2026 (SOI y OLR mensuales, TNI) quedaron vacíos.

---
---
---

# ¿Está el Litoral argentino bajo los efectos de El Niño a octubre de 2026?

*Borrador de informe para el TP grupal · Laboratorio de Métodos Cuantitativos, FCE-UBA, 2C 2026*

> Borrador redactado con asistencia de IA a partir del análisis en `analisis_el_nino/`. Versión 2 (16/10/2026): rehecha después de corregir los datos de 2026 del dataset del Pacífico, que no coincidían con NOAA (ver §3 y §6). La versión anterior, con la conclusión opuesta, quedó en `informe_el_nino/version_anterior_datos_erroneos/`. Antes de pasarlo a Overleaf o Word, el grupo debe contrastar cada cifra con el notebook (revisión intragrupal obligatoria).

## 1. Resumen

El trabajo evalúa si, a octubre de 2026, el Litoral argentino (Santa Fe, Entre Ríos, Corrientes y Misiones) se encuentra bajo los efectos del fenómeno de El Niño. Se combinaron índices mensuales del Pacífico tropical de NOAA (1950–2026) con observaciones diarias de 16 estaciones del Servicio Meteorológico Nacional (1991–julio 2026), completadas de agosto a mediados de octubre de 2026 con datos de NASA POWER. La respuesta se descompone en tres niveles: condición oceánica, acople atmosférico e impactos regionales. **La respuesta es sí, con un matiz.** El Niño está declarado por NOAA desde junio y es excepcionalmente intenso: el ONI de julio-septiembre (+2,16 °C) es el más alto registrado para ese trimestre desde 1950, y la atmósfera está acoplada (SOI significativamente negativo). En el Litoral ya aparecen señales compatibles: el trimestre julio-septiembre fue más lluvioso que el 94 % de los años 1991–2025. Río arriba, en las cuencas altas del Iguazú y del Uruguay en el sur de Brasil, llueve muy por encima de lo normal desde julio (julio-septiembre +64 % en el Iguazú alto, el tercer valor desde 1991), lo que explica las crecidas. El matiz es que en las estaciones del Litoral todavía no se observa el exceso sostenido de lluvia y de días extremos que caracteriza el impacto pleno: septiembre fue seco y la cantidad de días con lluvia extrema es normal. La relación histórica indica que ese impacto se concentra entre octubre y enero (r = 0,76 entre ONI y lluvia de octubre-diciembre), un período que recién comienza.

## 2. Introducción

El Niño-Oscilación del Sur (ENSO) es la principal fuente de variabilidad climática interanual en el sudeste de Sudamérica. Durante su fase cálida, el noreste argentino tiende a recibir más lluvia entre la primavera de un año y el verano siguiente (Ropelewski y Halpert, 1987; Grimm et al., 2000), con consecuencias sobre crecidas del Paraná y del Uruguay, la producción agropecuaria y la infraestructura.

La pregunta del trabajo es una foto al momento: **a octubre de 2026, ¿se puede afirmar que el Litoral argentino ya se encuentra bajo los efectos de El Niño?** No se busca pronosticar la evolución del fenómeno, ya que eso excede el contenido de la materia.

Para responderla se distinguen tres preguntas encadenadas, porque un impacto regional atribuible a El Niño requiere que las tres sean afirmativas:

1. **Condición oceánica:** ¿el Pacífico ecuatorial central está anómalamente cálido según el criterio oficial?
2. **Acople atmosférico:** ¿la atmósfera responde a ese calentamiento (Oscilación del Sur en fase negativa)?
3. **Impactos regionales:** ¿la lluvia del Litoral muestra la señal que históricamente se asocia a El Niño?

El marco de referencia es la tesis doctoral de Robledo (2012), que estudia los extremos diarios de precipitación en la Argentina y su relación con la temperatura del océano tropical, y de la que se toman los índices de extremos utilizados.

## 3. Datos y metodología

**Fuentes.**
- *Pacífico (NOAA):* `dataset_pacifico_final_limpio.csv`, 924 registros mensuales entre 1950 y 2026. Variables principales: ONI, anomalía de temperatura superficial en Niño 3.4, SOI y RONI. **Corrección:** los valores de enero a septiembre de 2026 del archivo original no coincidían con NOAA (por ejemplo, Niño 3.4 de septiembre −0,45 °C en el CSV contra +2,56 °C oficial). Se reemplazaron por los valores oficiales del Climate Prediction Center (CPC): Niño 3.4 y ONI de los archivos de índices del CPC, RONI de la actualización ENSO del 13/10/2026, y Niño 1+2 y Niño 3 de las discusiones ENSO mensuales. Lo que NOAA no publica con número quedó vacío. Los años 1950–2025 no cambiaron. El detalle está en `TP-Labo/DF/CAMBIOS_dataset_pacifico_2026.md`.
- *SMN:* datos diarios 1991–2020 (`.xlsx`) y desde 2021 (`.lst`), con temperatura y precipitación. El último dato disponible es del 31/07/2026.
- *NASA POWER, cuencas altas:* lluvia diaria de mayo a octubre de 1991–2026 en 14 puntos del sur de Brasil (cuencas altas del Iguazú y del Uruguay), en `datos_sep_oct_2026/cuencas_altas_nasapower_diario.csv`.
- *NASA POWER, Litoral:* lluvia y temperatura diarias en las coordenadas de las mismas 16 estaciones, del 1/8 al 12/10/2026, para cubrir el período sin datos del SMN. Es una grilla de ~50 km y no un pluviómetro, por eso las anomalías se calculan contra la climatología 1991–2020 de la propia NASA POWER. Validación contra el SMN en agosto-octubre de 1991–2020 (1419 meses-estación): la lluvia mensual correlaciona r = 0,95 y la anomalía regional r = 0,98, con el mismo signo en el 97 % de los meses. A escala diaria o de una estación la coincidencia es menor (diferencia típica de 9 mm por mes).

**Región de estudio.** Las 16 estaciones de Santa Fe, Entre Ríos, Corrientes y Misiones (Ceres, Reconquista, Sauce Viejo, Venado Tuerto, Rosario, El Trébol, Paraná, Concordia, Gualeguaychú, Corrientes, Paso de los Libres, Monte Caseros, Iguazú, Bernardo de Irigoyen, Posadas y Oberá).

**Transformaciones.**
1. Unificación de los dos archivos del SMN (formato, codificación latin-1, nombres de columnas distintos, filas sin fecha válida y duplicados).
2. Faltantes de precipitación: "S/D" como dato faltante y celda vacía como 0 mm, supuesto respaldado por una proporción realista de días con lluvia (~27 %).
3. Agregación mensual por estación, descartando estación-meses con menos del 90 % de días con dato.
4. Climatología 1991–2020 por estación y mes, y anomalía estandarizada (z). La serie regional es el promedio de las z de las estaciones. De agosto a octubre de 2026 la z sale de NASA POWER contra su propia climatología, y se empalma con la del SMN (empalme de anomalías, no de milímetros).
5. Índices de extremos de Robledo (2012): percentil 75 de los días con lluvia (P75), cantidad de días por encima de P75 (PE75) e intensidad media de esos días (DIER75).
6. Promedios trimestrales para alinear la lluvia con el ONI, y clasificación de fases desde el ONI (≥ +0,5 El Niño; ≤ −0,5 La Niña).

**Criterio de El Niño.** Se usan las dos definiciones de NOAA/CPC: la *operativa*, con la que el CPC declara un "El Niño Advisory" cuando el Pacífico supera +0,5 °C, se espera que persista y la atmósfera responde; y la *histórica*, que clasifica un episodio cuando el ONI es ≥ +0,5 °C durante al menos cinco trimestres consecutivos superpuestos.

**Herramientas.** Python (pandas, scipy, matplotlib) en Google Colab. Correlación de Pearson y prueba de Mann-Whitney para comparar fases.

## 4. Resultados

### 4.1 Condición oceánica: El Niño en curso, el más intenso registrado para esta época del año

El ONI pasó de −0,39 °C en diciembre-febrero (final de la La Niña 2025–26) a +0,95 °C en abril-junio, +1,39 °C en mayo-julio, +1,80 °C en junio-agosto y **+2,16 °C en julio-septiembre**. Son cuatro trimestres consecutivos por encima de +0,5 °C; el quinto (agosto-octubre) está asegurado, porque agosto y septiembre ya promedian más de +2 °C. La anomalía mensual de Niño 3.4 de septiembre es +2,56 °C. El CPC declaró El Niño ("El Niño Advisory") en su discusión del 11 de junio de 2026 y la mantiene vigente al 8 de octubre.

La comparación con eventos anteriores, que es lo que propone la idea del TP, es contundente. Ningún año desde 1950 tuvo un ONI tan alto en julio-septiembre: los máximos previos eran 2015 (+1,91 °C) y 1997 (+1,90 °C), los dos episodios más fuertes del período. Lo mismo vale para la anomalía de septiembre (2026: +2,56 °C; 2015: +2,21 °C; 1997: +2,13 °C). El calentamiento también es muy marcado sobre la costa sudamericana (Niño 1+2: +3,9 °C en septiembre). El RONI, que descuenta el calentamiento global de los trópicos, es +1,7 °C en julio-septiembre, de modo que la conclusión no depende de la tendencia de largo plazo.

*Figura 1. ONI 1991–2026 con umbrales de ±0,5 °C* (`analisis_el_nino/graficos/1_oni_linea_tiempo.png`).
*Figura 5. Trayectoria del ONI en 2026 frente a 1997, 2009, 2015 y 2023* (`5_trayectoria_oni_2026_vs_eventos.png`).

### 4.2 Acople atmosférico: la atmósfera responde

Durante El Niño, el SOI tiende a ser negativo. En la serie 1951–2025, ONI y SOI correlacionan en −0,72. En 2026 el SOI fue negativo en abril (−1,1) y mayo (−1,5), y según el CPC los índices de Oscilación del Sur tradicional y ecuatorial fueron "significativamente negativos" de julio a septiembre, cerca de −2 desvíos en julio. El dataset no tiene valores numéricos oficiales para junio a septiembre, por eso esas celdas quedaron vacías. Océano y atmósfera están acoplados, como corresponde a un evento maduro.

### 4.3 Impactos regionales: señales tempranas compatibles, sin exceso sostenido todavía

**La relación histórica es fuerte en primavera-verano y débil en invierno.** Entre 1991 y 2025, la correlación entre el ONI y la anomalía regional de lluvia del Litoral es de 0,76 en octubre-diciembre (OND), 0,75 en NDJ, 0,64 en DJF y 0,52 en SON, y cae a 0,27–0,36 entre junio y septiembre (no significativa en JAS). Los días de lluvia extrema siguen el mismo patrón: la correlación con PE75 en OND es 0,77 y, en primavera, los años El Niño tienen en promedio +0,37 días extremos por mes por encima de lo normal, contra −0,06 en años neutrales y −0,35 en años La Niña (Mann-Whitney El Niño vs resto: p = 0,02).

*Figura 2. Dispersión ONI vs lluvia del Litoral en OND, por fase* (`2_dispersion_oni_lluvia_OND.png`).
*Figura 3. Días de lluvia extrema (PE75) en primavera según fase ENSO* (`3_boxplot_PE75_por_fase.png`).

**2026: lluvia por encima de lo normal desde que se instaló El Niño.** Las anomalías regionales de enero a mayo alternan de signo (enero −0,96 z; mayo −0,71 z), lo esperable en el final de una La Niña. Desde junio, con El Niño ya declarado, predominan los meses lluviosos: junio +0,54, julio +1,25 (SMN), agosto +1,40, septiembre −0,49 y 1 al 12 de octubre +0,92 (NASA POWER). El trimestre julio-septiembre promedia +0,72 z, más lluvioso que el 94 % de los años 1991–2025 y por encima del promedio de los años El Niño en ese trimestre (+0,27). Agosto-octubre (con octubre parcial) promedia +0,61 z, percentil 86. Agosto fue muy lluvioso en el centro y sur del Litoral (Concordia 371 % de lo normal, Ceres 392 %) y el inicio de octubre es húmedo en Misiones y Corrientes.

**Pero todavía no es el impacto pleno.** Septiembre fue seco en 14 de las 16 estaciones (Concordia 29 % de lo normal). Los días de lluvia extrema entre el 1 de septiembre y el 12 de octubre son normales: 4,7 por estación contra 4,8 de la climatología (percentil 43; los años El Niño promedian +0,28 en esa ventana). Además, julio a septiembre es la parte del año en que la relación con ENSO es más débil, por lo que esas lluvias son *compatibles* con El Niño pero no alcanzan para atribuírselas. La ventana de mayor señal (octubre a enero) recién empieza y solo hay 12 días observados de octubre.

**Contexto fuera de los datasets.** Las observaciones coinciden con lo que reportan los organismos: el SMN da una probabilidad mayor al 55 % de lluvias superiores a lo normal en Misiones y Corrientes para los próximos meses, y el río Iguazú llegó a unos 12.000 m³/s a comienzos de octubre, alrededor de diez veces su caudal normal (La Nación, 5/10/2026). Estos datos no forman parte del análisis, pero son consistentes con él.

*Figura 4. Niño 3.4 y lluvia del Litoral, 2023–2026 (SMN hasta julio, NASA POWER desde agosto)* (`4_panel_reciente_2023_2026.png`).

### 4.4 Crecidas: la lluvia que cae río arriba

Las crecidas del Iguazú, el Paraná y el Uruguay no dependen solo de la lluvia en las estaciones del Litoral, sino de la que cae en sus cuencas altas, en el sur de Brasil. Por eso se armó un dataset complementario con NASA POWER: lluvia diaria de mayo a octubre, de 1991 a 2026, en 7 puntos de la cuenca alta del Iguazú (estado de Paraná) y 7 de la cuenca alta del Uruguay (Santa Catarina y Rio Grande do Sul). Las anomalías se calculan contra la climatología 1991–2020 de esos mismos puntos.

| Lluvia 2026, % de lo normal | May | Jun | Jul | Ago | Sep | 1–12 oct |
|---|---|---|---|---|---|---|
| Iguazú alto | 99 | 116 | 141 | 180 | 171 | 183 |
| Uruguay alto | 67 | 77 | 161 | 132 | 102 | 240 |

- **Cuenca alta del Iguazú:** julio a septiembre de 2026 acumuló 64 % más que lo normal (+1,52 z), el tercer valor más alto desde 1991 después de 1998 y 2009. Agosto-12 de octubre es el segundo más lluvioso de la serie. Esto explica que el río esté muy crecido aunque en la estación Iguazú, del lado argentino, la lluvia de agosto y septiembre haya sido normal (103 % y 90 %).
- **Cuenca alta del Uruguay:** julio fue muy lluvioso (161 %) y el 1 al 12 de octubre lleva 240 % de lo normal, el tercer comienzo de octubre más lluvioso desde 1991. Los dos que lo superan son 1997 y 2023, ambos años El Niño.

El resultado confirma que la crecida tiene origen en lluvias abundantes río arriba, sostenidas desde julio. La atribución a El Niño debe hacerse con cuidado: en estas cuencas, la lluvia de julio a septiembre casi no se correlaciona con el ONI en 1991–2025 (r ≈ 0), así que el invierno lluvioso es compatible con El Niño pero no se le puede atribuir. En octubre la relación aparece (r = 0,42 en la cuenca alta del Uruguay) y 2026 se ubica junto a los dos últimos El Niño fuertes, lo que sí es una señal temprana consistente con el evento. A esto se suma que el Litoral ya tenía suelos saturados por un junio-agosto lluvioso, por lo que lluvias moderadas alcanzan para producir anegamientos.

*Figura 6. Lluvia 2026 en las cuencas altas del Iguazú y del Uruguay* (`6_lluvia_cuencas_altas_2026.png`).

### 4.5 Síntesis por nivel

| Nivel | Indicador | Valor más reciente | ¿Compatible con El Niño? |
|---|---|---|---|
| Condición oceánica | ONI (JAS 2026) | +2,16 °C, récord para ese trimestre | Sí (4 trimestres ≥ +0,5; el 5.º asegurado) |
| Condición oceánica | Niño 3.4 (sep 2026) | +2,56 °C | Sí |
| Condición oceánica | Estado oficial CPC | El Niño Advisory desde junio 2026 | Sí |
| Acople atmosférico | SOI (jul–sep 2026) | Significativamente negativo (CPC) | Sí |
| Impacto regional | Lluvia Litoral, z JAS 2026 (SMN + NASA) | +0,72 (percentil 94) | Sí, señal temprana |
| Impacto regional | Lluvia Litoral, z septiembre 2026 | −0,49 | No |
| Impacto regional | Días extremos PE75, 1/9 al 12/10 | 4,7 vs 4,8 normal | No (todavía normal) |
| Cuencas altas (sur de Brasil) | Lluvia jul–sep, Iguazú alto | 164 % de lo normal (3.º desde 1991) | Explica la crecida; atribución a El Niño débil en invierno |
| Cuencas altas (sur de Brasil) | Lluvia 1–12 oct, Uruguay alto | 240 % (3.º, detrás de 1997 y 2023) | Sí, señal temprana |
| Relación histórica | r ONI vs lluvia OND (1991–2025) | 0,76 | La teleconexión es fuerte en primavera-verano |

## 5. Conclusiones

**Sí: a octubre de 2026 el Litoral argentino está bajo la influencia de El Niño, aunque su impacto en lluvias todavía está en una etapa temprana.** Los dos primeros niveles son inequívocos: el Pacífico atraviesa un El Niño declarado oficialmente, el más intenso registrado para esta época del año, y la atmósfera está acoplada. En el tercer nivel, la evidencia es favorable pero parcial. Desde que se instaló el evento, el Litoral tuvo un invierno y comienzo de primavera más lluviosos que en casi todos los años de la serie, y las cuencas altas del Iguazú y del Uruguay recibieron lluvias excepcionales que explican las crecidas actuales. Pero un septiembre seco y una cantidad normal de días extremos muestran que todavía no se observa el exceso sostenido que caracteriza a los años El Niño en primavera y verano.

La formulación más precisa para defender el trabajo es: **el forzante está presente y es muy fuerte, y la región ya muestra señales compatibles, pero con datos a mediados de octubre todavía no se puede demostrar con datos locales un impacto pleno.** Esto coincide con la hipótesis inicial del grupo ("muy probablemente sí, pero…").

**Limitaciones.**
- **Los datos de 2026 del dataset del Pacífico estaban mal y se corrigieron.** La versión anterior de este informe llegó a la conclusión opuesta por usarlos. Las celdas sin valor oficial numérico (por ejemplo, el SOI de junio a septiembre) quedaron vacías y el SOI se describe con el texto del CPC.
- Los datos del SMN terminan el 31/07/2026. Agosto a octubre se cubrieron con NASA POWER, que valida bien a escala mensual y regional, pero no reemplaza al pluviómetro en lluvias locales. Octubre tiene solo 12 días y los últimos días de NASA POWER son provisorios.
- La relación ENSO-lluvia es débil entre junio y septiembre, así que las lluvias de invierno, tanto en el Litoral como en las cuencas altas de Brasil, no se pueden atribuir a El Niño con los datos solos.
- Las cuencas altas se representan con 7 puntos de NASA POWER cada una, no con datos de caudal. Los caudales de los ríos (por ejemplo, del INA en Argentina o de la ANA en Brasil) serían la medición directa de la crecida.
- El período 1991–2025 incluye solo 11 primaveras con El Niño, y no todos los eventos mojan igual: en 2015, con ONI +2,5 °C, la lluvia de primavera fue normal (−0,03 z).
- La lluvia del Litoral también responde a otros forzantes (Dipolo del Índico, El Niño Modoki, Atlántico) no incluidos en los datos.
- La metodología adapta la de Robledo (2012), que usó 1961–2000 y descomposición SVD; aquí se usan el período 1991–2020 y correlaciones simples.

**Extensiones posibles.** Reemplazar NASA POWER por datos del SMN cuando se publiquen hasta octubre, repetir el análisis a fin de año con OND completo, incorporar caudales del Paraná, el Uruguay y el Iguazú, y evaluar pronósticos probabilísticos de ENSO en la materia correlativa.

## 6. Reflexión metodológica

Se utilizó IA (Claude) como asistente en la exploración de los datasets, la elección de transformaciones, la generación de gráficos y la redacción de este borrador. Las celdas del notebook asistidas por IA y los prompts utilizados deben indicarse en el Colab.

El episodio más importante del trabajo fue un error de datos. La primera versión del análisis concluyó que no había El Niño, porque el dataset del Pacífico mostraba un Niño 3.4 de −0,45 °C en septiembre de 2026. Al contrastarlo con los medios y con el boletín oficial del CPC, que informaba +2,1 °C y El Niño declarado, se comprobó que los valores de 2026 del CSV no eran los observados por NOAA: desde junio seguían una trayectoria casi idéntica a la de 2017 y estaban redondeados de forma sospechosa. Se reemplazaron por los valores oficiales, documentando cada cambio, y la conclusión se invirtió. La lección es que un dataset "limpio" puede traer errores en los datos más recientes, que son justamente los que responden una pregunta de actualidad, y que una conclusión que contradice el consenso de los organismos oficiales obliga a revisar los datos antes de defenderla. La IA detectó también otros problemas de calidad, como las columnas de fase ENSO asignadas por año calendario y las etiquetas trimestrales erróneas en 2025, lo que llevó a reclasificar las fases desde el ONI. La interpretación final y la validación de cada cifra contra el notebook son responsabilidad del grupo. *(Completar con los comentarios de la revisión intragrupal y, si se hace, de la validación externa.)*

## 7. Bibliografía y fuentes de datos

- Barros, V. y Silvestri, G. (2002). The relation between sea surface temperature at the subtropical South-Central Pacific and precipitation in southeastern South America. *Journal of Climate*, 15, 251–267.
- Grimm, A. M., Barros, V. R. y Doyle, M. E. (2000). Climate variability in southern South America associated with El Niño and La Niña events. *Journal of Climate*, 13, 35–58.
- La Nación (5/10/2026). *Súper Niño 2026: qué se sabe del fenómeno que cambiará el pronóstico de la Argentina.* https://www.lanacion.com.ar/sociedad/super-nino-2026-que-se-sabe-del-fenomeno-que-cambiara-por-completo-el-clima-en-la-argentina-y-nid05102026/
- NASA Langley Research Center. *POWER Project* (datos diarios de precipitación y temperatura). https://power.larc.nasa.gov/
- NOAA Climate Prediction Center. *ENSO Diagnostic Discussion* (junio a octubre de 2026). https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/enso_advisory/ensodisc.shtml
- NOAA Climate Prediction Center. *Índices ONI y Niño 3.4* (`oni.ascii.txt`, `detrend.nino34.ascii.txt`) y *ENSO: Recent Evolution, Current Status and Predictions* (13/10/2026). https://www.cpc.ncep.noaa.gov/data/indices/
- Robledo, F. A. (2012). *Extremos diarios de precipitación en la Argentina: cambios observados en la segunda mitad del siglo XX y asociación con la temperatura superficial del océano tropical.* Tesis doctoral, Facultad de Ciencias Exactas y Naturales, Universidad de Buenos Aires.
- Ropelewski, C. F. y Halpert, M. S. (1987). Global and regional scale precipitation patterns associated with the El Niño/Southern Oscillation. *Monthly Weather Review*, 115, 1606–1626.
- Servicio Meteorológico Nacional (SMN). Datos meteorológicos diarios de estaciones de superficie 1991–2026 y tabla de estaciones. https://www.smn.gob.ar/
- Vera, C., Silvestri, G., Barros, V. y Carril, A. (2004). Differences in El Niño response over the Southern Hemisphere. *Journal of Climate*, 17, 1741–1753.
